Load the 2-year preprocessed sales data from notebook 02 and train a Random Forest and Decision Tree regressor.

In [6]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

DATA_DIR = Path(os.getenv("IDX_DATA_DIR", "data"))
df = pd.read_csv(DATA_DIR / "processed/sales_preprocessed_2.csv")

train = df[df["split"] == "train"].drop(columns=["split"])
test = df[df["split"] == "test"].drop(columns=["split"])

y_train = train["ClosePrice"]
X_train = train.drop(columns=["ClosePrice"])
y_test = test["ClosePrice"]
X_test = test.drop(columns=["ClosePrice"])

y_train_log = np.log(y_train)
y_test_log = np.log(y_test)

print(X_train.shape, X_test.shape)

(241678, 601) (9849, 601)


In [8]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

tree = DecisionTreeRegressor(max_depth=12, min_samples_leaf=50, random_state=0)
tree.fit(X_train, y_train_log)

y_pred_tree_log = tree.predict(X_test)
y_pred_tree = np.exp(y_pred_tree_log)

print("decision tree ")
print("test RMSE (dollars):", np.sqrt(mean_squared_error(y_test, y_pred_tree)))
print("test R^2 (dollars):", r2_score(y_test, y_pred_tree))

decision tree 
test RMSE (dollars): 396929.84331410914
test R^2 (dollars): 0.7766261721734594


In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=0,
)
rf.fit(X_train, y_train_log)

y_pred_log = rf.predict(X_test)
y_pred = np.exp(y_pred_log)

print("random forest (log price target)")
print("train R^2 (log):", rf.score(X_train, y_train_log))
print("test R^2 (log):", r2_score(y_test_log, y_pred_log))
print("test RMSE (dollars):", np.sqrt(mean_squared_error(y_test, y_pred)))
print("test R^2 (dollars):", r2_score(y_test, y_pred))

random forest (log price target)
train R^2 (log): 0.9355834271897705
test R^2 (log): 0.9173855110423419
test RMSE (dollars): 305685.8191301644
test R^2 (dollars): 0.867518460762276


Redo both models on the 2-year target-encoded data from notebook 02_copy (City, PostalCode, HighSchoolDistrict means).

In [10]:
df_te = pd.read_csv(DATA_DIR / "processed/sales_preprocessed_1_target_encoded.csv")

train_te = df_te[df_te["split"] == "train"].drop(columns=["split"])
test_te = df_te[df_te["split"] == "test"].drop(columns=["split"])

y_train_te = train_te["ClosePrice"]
X_train_te = train_te.drop(columns=["ClosePrice"])
y_test_te = test_te["ClosePrice"]
X_test_te = test_te.drop(columns=["ClosePrice"])

y_train_te_log = np.log(y_train_te)
y_test_te_log = np.log(y_test_te)

print(X_train_te.shape, X_test_te.shape)

(120373, 114) (9827, 114)


In [11]:
tree_te = DecisionTreeRegressor(max_depth=12, min_samples_leaf=50, random_state=0)
tree_te.fit(X_train_te, y_train_te_log)

y_pred_tree_te = np.exp(tree_te.predict(X_test_te))

print("decision tree (target encoded)")
print("test RMSE (dollars):", np.sqrt(mean_squared_error(y_test_te, y_pred_tree_te)))
print("test R^2 (dollars):", r2_score(y_test_te, y_pred_tree_te))

decision tree (target encoded)
test RMSE (dollars): 335217.9692525492
test R^2 (dollars): 0.8401232446166378


In [ ]:
rf_te = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=0,
)
rf_te.fit(X_train_te, y_train_te_log)

y_pred_te_log = rf_te.predict(X_test_te)
y_pred_te = np.exp(y_pred_te_log)

print("random forest (target encoded)")
print("train R^2 (log):", rf_te.score(X_train_te, y_train_te_log))
print("test R^2 (log):", r2_score(y_test_te_log, y_pred_te_log))
print("test RMSE (dollars):", np.sqrt(mean_squared_error(y_test_te, y_pred_te)))
print("test R^2 (dollars):", r2_score(y_test_te, y_pred_te))

In [ ]:
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

residuals = y_test_te - y_pred_te
resid_line = LinearRegression().fit(y_pred_te.reshape(-1, 1), residuals)
x_line = np.linspace(y_pred_te.min(), y_pred_te.max(), 200)
y_line = resid_line.predict(x_line.reshape(-1, 1))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(y_pred_te, residuals, alpha=0.25, s=8, label="residuals")
ax.plot(x_line, y_line, color="crimson", linewidth=2, label="regression line")
ax.axhline(0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Predicted price")
ax.set_ylabel("Residual (actual − predicted)")
ax.set_title("Residuals vs predicted (target-encoded random forest)")
ax.legend()
plt.tight_layout()
plt.show()

Two-stage random forest on the target-encoded data: classify high-end vs typical from features, then route to a low-price or high-price forest.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

HIGH_CUTOFF = 2_000_000
rf_params = dict(n_estimators=100, max_depth=20, min_samples_leaf=20, n_jobs=-1, random_state=0)

y_high_train = (y_train_te > HIGH_CUTOFF).astype(int)

clf = RandomForestClassifier(**rf_params, class_weight="balanced")
clf.fit(X_train_te, y_high_train)

low_mask = y_train_te <= HIGH_CUTOFF
high_mask = y_train_te > HIGH_CUTOFF
rf_low = RandomForestRegressor(**rf_params)
rf_high = RandomForestRegressor(**rf_params)
rf_low.fit(X_train_te[low_mask], y_train_te_log[low_mask])
rf_high.fit(X_train_te[high_mask], y_train_te_log[high_mask])

y_high_pred = clf.predict(X_test_te)
y_pred_split_log = np.where(
    y_high_pred == 1,
    rf_high.predict(X_test_te),
    rf_low.predict(X_test_te),
)
y_pred_split = np.exp(y_pred_split_log)

low_test = y_test_te <= HIGH_CUTOFF
high_test = y_test_te > HIGH_CUTOFF

print("low houses")
print("test RMSE:", np.sqrt(mean_squared_error(y_test_te[low_test], y_pred_split[low_test])))
print("test R^2:", r2_score(y_test_te[low_test], y_pred_split[low_test]))
print("high houses")
print("test RMSE:", np.sqrt(mean_squared_error(y_test_te[high_test], y_pred_split[high_test])))
print("test R^2:", r2_score(y_test_te[high_test], y_pred_split[high_test]))

Grid-search the split model's two random forest regressors on train only (3-fold CV, log-price R²). The classifier is unchanged. This cell can take a while.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

HIGH_CUTOFF = 2_000_000
param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [12, 20],
    "min_samples_leaf": [10, 20],
}

y_high_train = (y_train_te > HIGH_CUTOFF).astype(int)
low_mask = y_train_te <= HIGH_CUTOFF
high_mask = y_train_te > HIGH_CUTOFF

clf = RandomForestClassifier(
    n_estimators=100, max_depth=20, min_samples_leaf=20,
    class_weight="balanced", n_jobs=-1, random_state=0,
)
clf.fit(X_train_te, y_high_train)

rf_base = RandomForestRegressor(n_jobs=-1, random_state=0)
grid_low = GridSearchCV(rf_base, param_grid, cv=3, scoring="r2", n_jobs=1, verbose=1)
grid_high = GridSearchCV(rf_base, param_grid, cv=3, scoring="r2", n_jobs=1, verbose=1)

grid_low.fit(X_train_te[low_mask], y_train_te_log[low_mask])
grid_high.fit(X_train_te[high_mask], y_train_te_log[high_mask])
print("best low params:", grid_low.best_params_, "cv R^2:", grid_low.best_score_)
print("best high params:", grid_high.best_params_, "cv R^2:", grid_high.best_score_)

y_high_pred = clf.predict(X_test_te)
y_pred_split = np.exp(np.where(
    y_high_pred == 1,
    grid_high.best_estimator_.predict(X_test_te),
    grid_low.best_estimator_.predict(X_test_te),
))

low_test = y_test_te <= HIGH_CUTOFF
high_test = y_test_te > HIGH_CUTOFF
print("low houses")
print("test RMSE:", np.sqrt(mean_squared_error(y_test_te[low_test], y_pred_split[low_test])))
print("test R^2:", r2_score(y_test_te[low_test], y_pred_split[low_test]))
print("high houses")
print("test RMSE:", np.sqrt(mean_squared_error(y_test_te[high_test], y_pred_split[high_test])))
print("test R^2:", r2_score(y_test_te[high_test], y_pred_split[high_test]))